# Building BRB systems from expert knowledge

Unlike black-box models that require training data to produce any output, BRB systems can be initialized entirely from **domain expert knowledge**. An expert specifies the rules and belief degrees based on experience, and the resulting model produces meaningful predictions immediately, no data required.

This notebook demonstrates:
1. Constructing a BRB from expert-defined rules (pipeline leak detection, Xu et al. 2007; Chen et al. 2011, Section 5.2)
2. The hybrid expert-then-train workflow (Yang et al. 2007)

The hybrid approach is central to the RIMER methodology (Yang et al. 2006): experts provide initial knowledge, and data refines it.

```
# requires: pip install matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from desdeo_brb import BRBModel, RuleBase
from desdeo_brb.utils import build_rule_antecedent_indices

## Pipeline leak detection

This is the canonical BRB application from the literature (Xu et al. 2007; Chen et al. 2011, Section 5.2, Table D.1). A pipeline monitoring system uses two measured inputs:

- **FlowDiff**: difference between expected and measured flow rate (8 referential values)
- **PressureDiff**: difference between expected and measured pressure (7 referential values)

The output is **LeakSize** with 5 referential values: {Zero (0), Small (2), Medium (4), High (6), VeryHigh (8)}.

The expert knows from experience:
- Large negative FlowDiff with dropping pressure → large leak
- Near-zero differences → normal operation (no leak)

Note that the two attributes have **different numbers** of referential values (8 vs 7).

In [ ]:
# Referential values (trained values from Chen et al. 2011, Section 5.2.3)
flow_diff_rv = np.array([-10.0, -4.1, -2.8, -1.79, -0.79, 0.25, 2.0, 3.0])
pressure_diff_rv = np.array([-0.01, -0.008, -0.005, 0.003, 0.0058, 0.008, 0.01])
leak_size_rv = np.array([0.0, 2.0, 4.0, 6.0, 8.0])

prv = [flow_diff_rv, pressure_diff_rv]

# Expert-defined belief degrees from Chen et al. 2011, Table D.1
# 56 rules (8 x 7), 5 consequents: [Zero, Small, Medium, High, VeryHigh]
expert_beliefs = np.array(
    [
        # FlowDiff = NL (most negative)
        [0.00, 0.00, 0.00, 0.00, 1.00],  # R1:  NL, NL -> VeryHigh leak
        [0.00, 0.00, 0.00, 0.30, 0.70],  # R2:  NL, NM
        [0.00, 0.00, 0.20, 0.80, 0.00],  # R3:  NL, NS
        [0.00, 0.00, 0.80, 0.20, 0.00],  # R4:  NL, Z
        [0.65, 0.35, 0.00, 0.00, 0.00],  # R5:  NL, PS
        [0.85, 0.15, 0.00, 0.00, 0.00],  # R6:  NL, PM
        [0.95, 0.05, 0.00, 0.00, 0.00],  # R7:  NL, PL
        # FlowDiff = NM
        [0.00, 0.00, 0.10, 0.90, 0.00],  # R8
        [0.00, 0.00, 0.70, 0.30, 0.00],  # R9
        [0.00, 0.70, 0.30, 0.00, 0.00],  # R10
        [0.00, 0.90, 0.10, 0.00, 0.00],  # R11
        [0.80, 0.20, 0.00, 0.00, 0.00],  # R12
        [0.90, 0.10, 0.00, 0.00, 0.00],  # R13
        [0.99, 0.01, 0.00, 0.00, 0.00],  # R14
        # FlowDiff = NS
        [0.00, 0.00, 0.40, 0.60, 0.00],  # R15
        [0.00, 0.00, 0.80, 0.20, 0.00],  # R16
        [0.00, 0.30, 0.60, 0.10, 0.00],  # R17
        [0.10, 0.80, 0.10, 0.00, 0.00],  # R18
        [0.90, 0.10, 0.00, 0.00, 0.00],  # R19
        [0.95, 0.05, 0.00, 0.00, 0.00],  # R20
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R21
        # FlowDiff = Z
        [0.00, 0.00, 0.50, 0.50, 0.00],  # R22
        [0.00, 0.10, 0.80, 0.10, 0.00],  # R23
        [0.00, 0.80, 0.20, 0.00, 0.00],  # R24
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R25: Z, Z -> Zero leak
        [0.95, 0.05, 0.00, 0.00, 0.00],  # R26
        [0.98, 0.02, 0.00, 0.00, 0.00],  # R27
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R28
        # FlowDiff = PS
        [0.00, 0.00, 0.60, 0.40, 0.00],  # R29
        [0.00, 0.20, 0.60, 0.20, 0.00],  # R30
        [0.10, 0.60, 0.30, 0.00, 0.00],  # R31
        [0.90, 0.10, 0.00, 0.00, 0.00],  # R32
        [0.95, 0.05, 0.00, 0.00, 0.00],  # R33
        [0.99, 0.01, 0.00, 0.00, 0.00],  # R34
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R35
        # FlowDiff = PM
        [0.00, 0.00, 0.60, 0.40, 0.00],  # R36
        [0.00, 0.20, 0.70, 0.10, 0.00],  # R37
        [0.00, 0.70, 0.30, 0.00, 0.00],  # R38
        [0.95, 0.05, 0.00, 0.00, 0.00],  # R39
        [0.99, 0.01, 0.00, 0.00, 0.00],  # R40
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R41
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R42
        # FlowDiff = PL
        [0.00, 0.10, 0.70, 0.20, 0.00],  # R43
        [0.00, 0.30, 0.60, 0.10, 0.00],  # R44
        [0.10, 0.70, 0.20, 0.00, 0.00],  # R45
        [0.98, 0.02, 0.00, 0.00, 0.00],  # R46
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R47
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R48
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R49
        # FlowDiff = PVL (most positive)
        [0.00, 0.10, 0.80, 0.10, 0.00],  # R50
        [0.00, 0.30, 0.70, 0.00, 0.00],  # R51
        [0.10, 0.80, 0.10, 0.00, 0.00],  # R52
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R53: PVL, Z -> Zero leak
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R54
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R55
        [1.00, 0.00, 0.00, 0.00, 0.00],  # R56
    ]
)

n_rules = 56
rule_base = RuleBase(
    precedent_referential_values=prv,
    consequent_referential_values=leak_size_rv,
    belief_degrees=expert_beliefs,
    rule_weights=np.full(n_rules, 1.0 / n_rules),
    attribute_weights=np.ones((n_rules, 2)),
    rule_antecedent_indices=build_rule_antecedent_indices(prv),
)

model = BRBModel(prv, leak_size_rv, rule_base=rule_base)
print(f"Rules: {model.rule_base.n_rules}")
print(f"FlowDiff referential values ({len(flow_diff_rv)}): {flow_diff_rv}")
print(f"PressureDiff referential values ({len(pressure_diff_rv)}): {pressure_diff_rv}")

In [ ]:
# Show a few representative rules using the built-in formatter
print(rule_base.describe_all_rules(
    attribute_names=['FlowDiff', 'PressureDiff'],
    consequent_name='LeakSize',
))

In [ ]:
# Predict at several scenarios
scenarios = {
    "Normal operation": [-0.79, 0.003],
    "Small leak": [-2.8, -0.005],
    "Large leak": [-10.0, -0.01],
}

print("Scenario predictions (no training data needed):")
for name, (fd, pd) in scenarios.items():
    X = np.array([[fd, pd]])
    y = model.predict_values(X)[0]
    print(
        f"  {name:20s} (FlowDiff={fd:+6.2f}, PressureDiff={pd:+.4f}) -> LeakSize = {y:.2f}"
    )

In [ ]:
# Plot LeakSize vs FlowDiff at fixed PressureDiff values
flow_range = np.linspace(-10, 3, 200)
press_values = [-0.008, -0.005, 0.003, 0.008]

plt.figure(figsize=(8, 5))
for pd in press_values:
    X = np.column_stack([flow_range, np.full(200, pd)])
    y = model.predict_values(X)
    plt.plot(flow_range, y, label=f"PressureDiff={pd}")

plt.xlabel("FlowDiff")
plt.ylabel("Predicted LeakSize")
plt.title("Expert BRB: LeakSize vs FlowDiff")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The hybrid approach: expert initialization + data refinement

In practice, the expert's initial beliefs may be approximate. Training with operational data can fine-tune the belief degrees, rule weights, and attribute weights while preserving the structural constraints of the BRB. This is the optimization approach described by **Yang et al. (2007)**.

We demonstrate this with a simpler example: modeling $f(x) = \sin(2\pi x)$ on $[0, 1]$, starting with deliberately wrong (uniform) beliefs.

In [ ]:
# Setup: uniform (wrong) expert beliefs
prv_sin = [np.array([0.0, 0.25, 0.5, 0.75, 1.0])]
crv_sin = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
n_rules_sin = 5
n_cons = 5

uniform_beliefs = np.full((n_rules_sin, n_cons), 1.0 / n_cons)

rb_sin = RuleBase(
    precedent_referential_values=prv_sin,
    consequent_referential_values=crv_sin,
    belief_degrees=uniform_beliefs,
    rule_weights=np.full(n_rules_sin, 1.0 / n_rules_sin),
    attribute_weights=np.ones((n_rules_sin, 1)),
    rule_antecedent_indices=build_rule_antecedent_indices(prv_sin),
)

model_sin = BRBModel(prv_sin, crv_sin, rule_base=rb_sin)

# Evaluate before training
X_eval = np.linspace(0, 1, 200).reshape(-1, 1)
y_true = np.sin(2 * np.pi * X_eval[:, 0])
y_before = model_sin.predict_values(X_eval)

# Train
rng = np.random.default_rng(67)
X_train = rng.uniform(0, 1, size=(200, 1))
y_train = np.sin(2 * np.pi * X_train[:, 0])
model_sin.fit(X_train, y_train, fix_endpoints=True, n_restarts=3, verbose=True)

y_after = model_sin.predict_values(X_eval)

print(f"MSE before training: {np.mean((y_true - y_before) ** 2):.4f}")
print(f"MSE after training:  {np.mean((y_true - y_after) ** 2):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(X_eval[:, 0], y_true, "k-", linewidth=2, label="True $\\sin(2\\pi x)$")
axes[0].plot(
    X_eval[:, 0], y_before, "r--", linewidth=1.5, label="Uniform beliefs (wrong)"
)
axes[0].set_title("Before training")
axes[0].set_xlabel("x")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(X_eval[:, 0], y_true, "k-", linewidth=2, label="True $\\sin(2\\pi x)$")
axes[1].plot(X_eval[:, 0], y_after, "b-", linewidth=1.5, label="Trained BRB")
axes[1].set_title("After training")
axes[1].set_xlabel("x")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Verify structural constraints are preserved after training
rb_trained = model_sin.rule_base
print("Belief degree row sums:", np.round(rb_trained.belief_degrees.sum(axis=1), 6))
print("Rule weight sum:", round(rb_trained.rule_weights.sum(), 6))
print("Attribute weights >= 0:", np.all(rb_trained.attribute_weights >= 0))
print(
    "Referential values sorted:",
    np.all(np.diff(rb_trained.precedent_referential_values[0]) >= 0),
)

## Contrast with black-box models

A neural network needs training data before it can produce any output at all. A BRB system, by contrast, produces **physically meaningful outputs from expert knowledge alone** (as shown in the pipeline example above), and data only improves the accuracy.

This makes BRB particularly suitable for domains where:
- Data is scarce or expensive to collect
- Expert knowledge is available and reliable
- Interpretability is required (e.g., safety-critical systems)

Applications in the literature include clinical risk assessment (Kong et al. 2016), marine diesel fault diagnosis (Xu et al. 2018), and preference modeling in multi-objective optimization (Misitano 2020, using the INFRINGER method).

## Summary

This notebook demonstrated:
1. Constructing a BRB from expert-defined rules (no data needed)
2. Making predictions with the expert model
3. The hybrid expert-then-train workflow
4. Verification that structural constraints are preserved after training

**Next:** See `04_explainability.ipynb` for a deep dive into how BRB inference can be explained and visualized.